# Writing Bake-off: Round 2

Seven configs, two briefs, 28 calls. Effort matrix on Opus 5 and Sonnet 5.

Everything persists to `./local-data/writing_bakeoff/`.

**Run order:** Setup → Sources → Probe → Run → Blind → Grade → (score by hand) → Reveal.

**What this measures.** When both briefs list every banned word and structure, models avoid all of them, so the mechanical fault columns come back flat. That is prompt contamination, not a model result. The human columns (voice drift, gap admission, length compliance) carry the decision; the mechanical counters stay as a regression check. Short-form gets three reps so a single lucky post cannot decide a lane.

**Two things verified against OpenRouter docs before writing this:**

- `reasoning.effort` accepts `max`, `xhigh`, `high`, `medium`, `low`, `minimal`, `none`.
- `anthropic/claude-sonnet-5` is the current slug; Sonnet 5 shipped June 30 2026 and supersedes 4.6.

Slugs are still unverified guesses until the probe cell passes. Do not skip it.

**Fill the placeholders first.** Brief B contains `<...>` placeholders and a `[Placeholder ...]` persona
marker. Replace them with your own material before you run.

## 1. Setup

In [ ]:
import os, sys, subprocess

try:
    import httpx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    import httpx

from pathlib import Path

VOLUME = Path("./local-data")
ROOT = VOLUME / "writing_bakeoff"
SOURCES = ROOT / "sources"
OUTPUTS = ROOT / "outputs"
for d in (ROOT, SOURCES, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

RAW = ROOT / "raw_results.jsonl"
KEY = ROOT / "key.json"
SCORES = ROOT / "scores.csv"
REPORT = ROOT / "grading_report.md"

KEY_ENV = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not KEY_ENV:
    print("OPENROUTER_API_KEY not found.\n"
          "Attach a secret containing it in the notebook sidebar, or set it here:\n"
          "  os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-...'")
else:
    print(f"key loaded, writing to {ROOT}")

## 2. Config

Rates are USD per million tokens, from the confirmed price table in the next cell (source and
retrieval date are listed there). Cost is computed from token counts rather than trusted from
the response, because a provider can report `usage.cost` as `0`.

Introductory rates expire. Check the current listed price before you compare cost.

In [ ]:
MODELS = {
    "opus-5-med":     {"slug": "anthropic/claude-opus-5",          "effort": "medium", "in": 10.00, "out": 50.00},
    "opus-5-high":    {"slug": "anthropic/claude-opus-5",          "effort": "high",   "in": 10.00, "out": 50.00},
    "sonnet-5-low":   {"slug": "anthropic/claude-sonnet-5",        "effort": "low",    "in": 2.20,  "out": 11.00},
    "sonnet-5-med":   {"slug": "anthropic/claude-sonnet-5",        "effort": "medium", "in": 2.20,  "out": 11.00},
    "haiku-4.5":      {"slug": "anthropic/claude-haiku-4.5",       "effort": None,     "in": 1.10,  "out": 5.50},
    "inkling-xhigh":  {"slug": "thinkingmachines/inkling",         "effort": "xhigh",  "in": 1.00,  "out": 4.05},
    "luna-med":       {"slug": "openai/gpt-5.6-luna",              "effort": "medium", "in": 0.80,  "out": 3.60},
}

# One post per config cannot tell a real edge from a lucky draw, so short-form gets three reps.
# Long-form stays at one: each report is minutes of wall clock and the reads gate grading anyway.
# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries come from the confirmed price table below. A model without a
# confirmed price makes the notebook refuse to run, before any paid call.
from datetime import datetime, timezone
import math, os, re
from pathlib import Path


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)   # one cap per run; override with the RUN_BUDGET_USD env var
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output). Source: OpenRouter endpoint listings
# https://openrouter.ai/api/v1/models/<slug>/endpoints (haiku via the same listing), retrieved 2026-11-05.
# Each figure is the highest price across the listed endpoints and tiers, so reservations are worst case.
PRICE = {
    "anthropic/claude-opus-5":    (10.00, 50.00),
    "anthropic/claude-sonnet-5":  (2.20, 11.00),
    "anthropic/claude-haiku-4.5": (1.10, 5.50),
    "thinkingmachines/inkling":   (1.00, 4.05),
    "openai/gpt-5.6-luna":        (0.80, 3.60),
}
missing_prices = sorted(set([m["slug"] for m in MODELS.values()]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"Refusing to run: no confirmed price for {missing_prices}. Add the model's confirmed listed price "
                       "(USD per million tokens, input and output) to PRICE in this cell, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through the two checks below,
# which refuse an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if not isinstance(RUN_ID, str) or not RUN_ID.strip():
    raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {RUN_ID!r}).")
if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", RUN_ID):
    raise RuntimeError(f"RUN_ID {RUN_ID!r} is not a safe directory name. Refusing to start.")


LEDGER_ROOT = _ledger_root()
RUN_DIR = os.path.join(str(LEDGER_ROOT), RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file in the shared ledger root: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete LEDGER_ROOT/runner.lock
# by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(str(LEDGER_ROOT), "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner (any notebook) may be running, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


REPS = {"B-shortform": 3, "A-longform": 1}

MAX_TOKENS = 32000   # reasoning tokens come out of this budget; a small cap can starve a reasoning model to empty output
TIMEOUT_S  = 900
WORKERS    = 14
RETRIES    = 3

TOTAL_CALLS = sum(REPS[b] for _ in MODELS for b in REPS)
print(f"{len(MODELS)} configs x {sum(REPS.values())} runs = {TOTAL_CALLS} calls, {WORKERS} workers, cap ${RUN_BUDGET_USD:.2f}, ledger {LEDGER_PATH}")

## 3. Sources

Pulled live from the Federal Register API so the notebook is self-contained.

Both documents post-date every tested model's training cutoff, so a model cannot answer
from memory and has to work from the source in the prompt.

In [ ]:
import re, html, json
import httpx

FR_DOCS = {
    "fsis_salmonella_initiative_2026-14216.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/15/2026-14216.txt",
    "fda_citrus_red_2_revocation_2026-14909.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/23/2026-14909.txt",
}

def clean(t):
    t = re.sub(r"(?s)<head>.*?</head>", "", t)
    t = re.sub(r"<[^>]+>", "", t)
    return re.sub(r"\n{3,}", "\n\n", html.unescape(t)).strip()

for name, url in FR_DOCS.items():
    p = SOURCES / name
    if p.exists():
        print(f"  cached  {name}: {len(p.read_text().split())} words")
        continue
    txt = clean(httpx.get(url, timeout=90).text)
    p.write_text(txt, encoding="utf-8")
    print(f"  fetched {name}: {len(txt.split())} words")

SOURCE_TEXT = "\n\n---\n\n".join(
    f"[SOURCE FILE: {p.name}]\n\n{p.read_text(encoding='utf-8')}"
    for p in sorted(SOURCES.glob("*.txt"))
)
print(f"\ncombined source: {len(SOURCE_TEXT.split())} words")

## 4. Briefs

Read these before grading. You score `fabrications` and `citation_failures` against the
sources, so you need to know what they say.

In [ ]:
BRIEF_A = """You are writing a regulatory intelligence report for a client of a regulatory consultancy. The client is an industry association. The report goes out under the analyst's name and gets read by the client's executive director and passed to its member companies.

Write the report from the source material below. Do not use anything outside it.

## What the report has to do

Explain what happened, what it requires of member companies, and what to watch next. The reader knows food safety; they do not know this specific document. They need to finish reading and know what to do on Monday.

Length: 2,000 words or more.

## Structure

Open with what happened and why the reader is getting this. No preamble about the importance of food safety.

Then the body, organized however the material actually organizes itself. Do not force a template onto it.

Close with what to watch next. Specific things: dates, comment periods, follow-on actions, named agencies.

## Citation

Every factual claim traces to the source material below. Cite inline by document name and section or paragraph, like: (FSIS Recall 012-2026, para. 4).

If the source does not answer a question the report needs answered, write that the source does not address it. Do not fill the gap with what is usually true. An analyst who invents a number to make a report feel complete is worse than one who leaves a hole marked.

## Voice

Plain and direct. Sentences under 16 words where the meaning allows it.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No "it's not X, it's Y." No "despite this." No "but here's the thing."

No sentence that tells the reader something is important. Show what it requires and let them judge.

No summary ending. No "in conclusion," no "overall," no wrap-up paragraph that restates the report.

No vague authority. No "experts say," no "industry reports suggest." Name the source or cut the claim.

Semicolons rather than em-dashes.

Any statistic gets a date and a named source, or it comes out.

## Output

The report only. No cover note, no explanation of your approach, no offer to revise.

---

# SOURCE MATERIAL

""" + SOURCE_TEXT

BRIEF_B = """Write a LinkedIn post as the persona described below.

## Who you are

You are a professional writer with ten years of experience in B2B software. [Placeholder persona: replace with your own.]

## What to write about

He has run a benchmark on which AI model to use for regulatory work. <your benchmark: how many calls, across how many models>. The finding that surprised him: <your headline finding>. <two or three supporting numbers from your benchmark, each with its unit>.

<your own cost comparison, for example the cheapest and the most expensive model you tested and where each landed; or delete this paragraph>

His conclusion: <your conclusion, in one or two sentences>. And a human still reads everything before it ships.

## Shape

Hook, then receipts, then principle. That order.

The hook is a real observation, not a question and not a statistic dressed as a hook. No "Here's what I learned." No "I ran an experiment. The results shocked me."

The receipts are the numbers above. Use them. They are his own and they are dated <date of your benchmark>.

The principle lands at the end and does not preach. Write it in your own words; do not reuse the phrasing above.

Length: 150 to 250 words.

## Voice rules

Lowercase is fine. No hashtags. No engagement bait; no "thoughts?", no "agree?", no "drop a comment."

Semicolons rather than em-dashes.

Sentences under 16 words.

Vary the rhythm. Do not put every sentence on its own line; that reads as machine-generated even when each line is fine.

He does not say "quick one," "biz," "folks in the space," or "let's dive in." He does not repeat a person's name back at them. He does not sound like a tech bro.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No "it's not X, it's Y." No "not A. Not B. But C." No "despite this."

No cringe transition questions. No "The catch?" No "The kicker?" No "Sound familiar?"

No invented people. No "a QA manager I know named Pat."

No importance sentences. Nothing about what this means for the broader industry.

No summary ending.

He does not claim things he has not measured. Everything in this post is from the benchmark described above; do not add outside statistics, and do not name specific model vendors that are not in the numbers above.

## Output

The post only. No caption, no alternate versions, no explanation."""

BRIEFS = {"A-longform": BRIEF_A, "B-shortform": BRIEF_B}
for k, v in BRIEFS.items():
    print(f"{k}: {len(v.split())} words of prompt")

## 5. Probe

One short call per config. A bad slug 404s here for pennies instead of 28 wasted calls.
Seven configs is seven chances for a bad slug.

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=300, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = httpx.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def headers():
    return {
        "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://example.com/labs-benchmark",
        "X-Title": "labs-benchmark",
    }

def payload_for(cfg_name, messages, max_tokens):
    cfg = MODELS[cfg_name]
    p = {
        "model": cfg["slug"],
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 1.0,
        "usage": {"include": True},   # ask for cost explicitly; a provider can otherwise report 0
    }
    if cfg["effort"]:
        p["reasoning"] = {"effort": cfg["effort"]}
    return p

def price(cfg_name, ptok, ctok):
    cfg = MODELS[cfg_name]
    if ptok is None or ctok is None:
        return None
    pin, pout = PRICE[cfg["slug"]]
    return (ptok / 1e6) * pin + (ctok / 1e6) * pout

def call(cfg_name, messages, max_tokens=None):
    max_tokens = max_tokens or MAX_TOKENS
    p = payload_for(cfg_name, messages, max_tokens)
    last = None
    for attempt in range(RETRIES):
        t0 = time.time()
        try:
            r = guarded_post(p, headers(), timeout=TIMEOUT_S, purpose=cfg_name)
            el = round(time.time() - t0, 2)
            if r.status_code == 400:
                return {"ok": False, "error": f"400 {r.text[:400]}", "latency_s": el}
            r.raise_for_status()
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:400], "latency_s": el}

            ch = d["choices"][0]
            text = ch["message"].get("content")
            finish = ch.get("finish_reason")
            u = d.get("usage") or {}
            ctok = u.get("completion_tokens")
            ptok = u.get("prompt_tokens")

            # A reasoning model at high effort can spend the whole budget thinking and
            # return content=None. Logged as ok=true, it would have been
            # graded as a fault-free output. It is a failure; retry it bigger.
            if not text or not text.strip():
                last = (f"empty content (finish={finish}, completion_tokens={ctok}); "
                        f"budget consumed by reasoning at effort={MODELS[cfg_name]['effort']}")
                if attempt < RETRIES - 1:
                    max_tokens = int(max_tokens * 1.5)
                    p = payload_for(cfg_name, messages, max_tokens)
                    continue
                return {"ok": False, "error": last, "latency_s": el,
                        "finish_reason": finish, "completion_tokens": ctok}

            return {
                "ok": True, "text": text, "latency_s": el,
                "finish_reason": finish, "truncated": finish == "length",
                "prompt_tokens": ptok, "completion_tokens": ctok,
                "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens"),
                "cost_reported": u.get("cost"),
                "cost_computed": price(cfg_name, ptok, ctok),
                "provider": d.get("provider"),
            }
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            if attempt < RETRIES - 1:
                time.sleep(2 ** attempt)
    return {"ok": False, "error": last or "unknown", "latency_s": None}


msgs = [{"role": "user", "content": "Reply with the single word: ok"}]
bad = []
for name in MODELS:
    res = call(name, msgs, max_tokens=2000)
    if res["ok"]:
        rep = res.get("cost_reported")
        comp = res.get("cost_computed")
        print(f"  OK    {name:16s} {MODELS[name]['slug']:34s} {res['latency_s']:>6}s  "
              f"reported={rep}  computed=${comp:.6f}" if comp else
              f"  OK    {name:16s} {MODELS[name]['slug']:34s} {res['latency_s']:>6}s")
    else:
        bad.append((name, res["error"]))
        print(f"  FAIL  {name:16s} {MODELS[name]['slug']:34s} {res['error'][:100]}")

if bad:
    print("\n" + "=" * 70)
    print("Fix these in MODELS before running. Real slugs: https://openrouter.ai/models")
    for n, e in bad:
        print(f"\n  {n}\n  {e[:300]}")
else:
    print("\nAll seven resolved, and every effort value was accepted (a rejected effort")
    print("would have 400'd here). Safe to run.")

## 6. Run

28 calls, resume-safe. Re-running skips whatever already succeeded, so a dropped
kernel costs you only the in-flight calls.

In [ ]:
def load_done():
    done = set()
    if RAW.exists():
        for line in RAW.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            t = r.get("text")
            if r.get("ok") and isinstance(t, str) and t.strip():
                done.add((r["model"], r["brief"], r.get("rep", 1)))
    return done

done = load_done()
jobs = [(m, b, rep)
        for m in MODELS for b in BRIEFS
        for rep in range(1, REPS[b] + 1)
        if (m, b, rep) not in done]

print(f"{len(jobs)} to run, {len(done)} already done.\n")

def work(job):
    m, b, rep = job
    res = call(m, [{"role": "user", "content": BRIEFS[b]}])
    res.update({"model": m, "brief": b, "rep": rep,
                "slug": MODELS[m]["slug"], "effort": MODELS[m]["effort"],
                "ts": datetime.now(timezone.utc).isoformat()})
    return res

if jobs:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = {pool.submit(work, j): j for j in jobs}
        with RAW.open("a", encoding="utf-8") as fh:
            for fut in as_completed(futs):
                r = fut.result()
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
                fh.flush()
                tag = f"{r['model']}#{r['rep']}"
                if r["ok"]:
                    print(f"  done  {tag:20s} {r['brief']:12s} {len(r['text'].split()):>5}w  "
                          f"{r['latency_s']:>6}s  ${r.get('cost_computed') or 0:.4f}")
                else:
                    print(f"  FAIL  {tag:20s} {r['brief']:12s} {r['error'][:70]}")

print(f"\n{len(load_done())} of {TOTAL_CALLS} complete.")

## 7. Blind

Strips model identity, shuffles within each brief, writes `outputs/` plus `key.json`.

**Do not open `key.json` until every score is entered.** Word count is a fingerprint,
so do not run `wc -w` on the outputs either.

In [ ]:
import hashlib, random

SCRUB = [
    (re.compile(r"\b(Claude|Anthropic|OpenAI|ChatGPT|GPT-?\d[\w.\-]*|Gemini|Sonnet|Opus|"
                r"Haiku|Luna|Sol|Terra|Inkling|Grok|Thinking Machines)\b", re.I), "[MODEL]"),
    (re.compile(r"^\s*(As an AI|I'm an AI|As a language model)[^\n]*\n?", re.I | re.M), ""),
]

def scrub(t):
    for pat, rep in SCRUB:
        t = pat.sub(rep, t)
    return t.strip()

records = []
for line in RAW.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    r = json.loads(line)
    t = r.get("text")
    if r.get("ok") and isinstance(t, str) and t.strip():
        records.append(r)

latest = {}
for r in records:
    latest[(r["model"], r["brief"], r.get("rep", 1))] = r
records = list(latest.values())

if len(records) < TOTAL_CALLS:
    have = set(latest)
    missing = [f"{m}/{b}#{rp}" for m in MODELS for b in BRIEFS
               for rp in range(1, REPS[b] + 1) if (m, b, rp) not in have]
    print(f"WARNING: {len(records)}/{TOTAL_CALLS}. Missing: {', '.join(missing)}\n")

trunc = [r for r in records if r.get("truncated")]
if trunc:
    print("WARNING: hit the token ceiling, may be cut off mid-report:")
    for r in trunc:
        print(f"  {r['model']}/{r['brief']}#{r.get('rep',1)} ({r.get('completion_tokens')} ctok)")
    print("A truncated output scores well on faults purely by being short. Rerun those.\n")

for f in OUTPUTS.glob("*.md"):
    f.unlink()

random.seed(None)
key, by_brief = {}, {}
for r in records:
    by_brief.setdefault(r["brief"], []).append(r)

for brief, recs in by_brief.items():
    random.shuffle(recs)
    for i, r in enumerate(recs, 1):
        label = f"{brief}_{i:02d}"
        body = scrub(r["text"])
        (OUTPUTS / f"{label}.md").write_text(body, encoding="utf-8")
        key[label] = {
            "model": r["model"], "slug": r["slug"], "effort": r["effort"],
            "rep": r.get("rep", 1), "latency_s": r.get("latency_s"),
            "prompt_tokens": r.get("prompt_tokens"),
            "completion_tokens": r.get("completion_tokens"),
            "reasoning_tokens": r.get("reasoning_tokens"),
            "cost_reported": r.get("cost_reported"),
            "cost_computed": r.get("cost_computed"),
            "words": len(r["text"].split()),
            "sha1": hashlib.sha1(body.encode()).hexdigest()[:12],
        }

KEY.write_text(json.dumps(key, indent=2), encoding="utf-8")
print(f"{len(records)} outputs written blind to {OUTPUTS}")
print(f"key -> {KEY}   (do not open it yet)")

## 8. Grade — machine columns

Counts what a regex settles, and pre-screens fabrications by diffing every number,
date, docket, CFR cite and OMB number against the sources by word boundary.

Expect most of these to come back zero. Both briefs name the banned words and structures
explicitly, so the models avoid them; that is prompt contamination, and it is why the
human columns carry the decision. The counters stay as a regression check.

In [ ]:
import csv
from collections import Counter

AI_VOCAB = ["pivotal","crucial","enhance","enhances","enhanced","enhancing","underscore",
    "underscores","underscoring","vibrant","testament","foster","fosters","fostering",
    "delve","delves","delving","showcase","showcases","showcasing","intricate","evolving",
    "cultivate","cultivates","crafting","utilize","utilizes","utilizing","utilization",
    "tapestry","garner","garners","encompass","encompasses","encompassing","interplay",
    "myriad","realm","unsung hero","secret weapon"]

CONTEXTUAL = [
    (re.compile(r"\b(regulatory|compliance|business|competitive|industry|broader|shifting|"
                r"changing|current)\s+landscape\b", re.I), "landscape (abstract)"),
    (re.compile(r"\bexecute\s+(on|against)\b", re.I), "execute (verb)"),
]

CONTRAST = [
    (re.compile(r"\b(it'?s|this is|that'?s)\s+not\s+[^.;:!?]{2,60}?[,;]\s*(it'?s|but|that'?s)\b", re.I), "it's not X, it's Y"),
    (re.compile(r"\bnot\s+(just|only|merely|simply)\s+[^.;:!?]{2,60}?\bbut\b", re.I), "not just X but Y"),
    (re.compile(r"\bdespite\s+(this|that|these|those)\b", re.I), "despite this"),
    (re.compile(r"\bbut\s+here'?s\s+the\s+(thing|kicker|catch|problem)\b", re.I), "but here's the thing"),
    (re.compile(r"(?m)^\s*Not\s+[^.\n]{2,50}\.\s*Not\s+[^.\n]{2,50}\.", re.I), "Not A. Not B."),
    (re.compile(r"\bless\s+about\s+[^.;:!?]{2,60}?\bmore\s+about\b", re.I), "less about X, more about Y"),
]

IMPORTANCE = [
    (re.compile(r"\b(represents?|marks?|signals?)\s+a\s+(significant|major|important|fundamental)\b", re.I), "represents a significant"),
    (re.compile(r"\bthe\s+(broader|wider)\s+(implications?|context|trend|shift|picture)\b", re.I), "the broader X"),
    (re.compile(r"\b(underscores?|highlights?|reflects?)\s+(the|a)\s+(importance|significance|need)\b", re.I), "underscores the importance"),
    (re.compile(r"\bcannot\s+be\s+(overstated|understated)\b", re.I), "cannot be overstated"),
    (re.compile(r"\bin\s+an\s+(era|age)\s+(of|where)\b", re.I), "in an era of"),
]

SUMMARY = [
    # Anchored to a line OR a sentence start; line-start alone missed ". In conclusion, ..."
    (re.compile(r"(?mi)(?:^|(?<=[.!?])\s)\s*(?:#+\s*)?(in\s+conclusion|to\s+conclude|"
                r"in\s+summary|to\s+sum\s+up|overall|ultimately|in\s+closing|"
                r"the\s+bottom\s+line|final\s+thoughts?|key\s+takeaways?)\b"), "summary opener"),
    (re.compile(r"\bat\s+the\s+end\s+of\s+the\s+day\s*,", re.I), "at the end of the day,"),
]

VAGUE = [
    (re.compile(r"\bexperts?\s+(say|believe|agree|note|suggest|warn)\b", re.I), "experts say"),
    (re.compile(r"\b(industry|market|recent)\s+(reports?|analysts?|data)\s+(say|suggest|show|indicate)\b", re.I), "industry reports"),
    (re.compile(r"\b(many|most|some)\s+(believe|argue|say|think|agree)\b", re.I), "many believe"),
    (re.compile(r"\b(studies|research)\s+(show|suggest|indicate)\b", re.I), "studies show"),
]

CRINGE = [
    (re.compile(r"(?m)^\s*(the\s+)?(catch|kicker|twist|result|problem)\?\s*$", re.I), "The catch?"),
    (re.compile(r"\bsound\s+familiar\?", re.I), "Sound familiar?"),
    (re.compile(r"(?m)^\s*(and\s+)?the\s+(best|worst|craziest)\s+part\?\s*$", re.I), "The best part?"),
    (re.compile(r"\blet\s+that\s+sink\s+in\b", re.I), "let that sink in"),
    (re.compile(r"\bhere'?s\s+what\s+(i|we)\s+(learned|found|discovered)\b", re.I), "here's what I learned"),
]

FAKE = re.compile(
    r"\b(?:a|one|our|my)\s+(?:QA|quality|plant|food safety|compliance)\s+"
    r"(?:manager|director|lead|supervisor)\s+(?:named|called)\s+[A-Z][a-z]+|"
    r"\b(?:Sarah|John|Jane|Mike|Emily|David|Lisa|Maria)\s+"
    r"(?:Chen|Smith|Johnson|Williams|Martinez|Patel|Kim|Lee)\b")

STAT = re.compile(r"(?<![\w.$])(\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*(?:%|percent)|\$\s?\d[\d,.]*\s*(?:million|billion|k|m|b)?)", re.I)
# A month name counts as a date only next to a day or year; without that guard
# "costs may reach" reads as a date and clears the stat next to it.
DATE_NEAR = re.compile(r"(?:19|20)\d{2}"
    r"|\b(?:January|February|March|April|May|June|July|August|September|October|November|December"
    r"|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sept?|Oct|Nov|Dec)\.?\s+\d{1,2}\b"
    r"|\bQ[1-4]\s*(?:19|20)?\d{2}\b", re.I)
SOURCE_NEAR = re.compile(r"\b(FSIS|FDA|USDA|CDC|EPA|Federal Register|\d{1,2}\s*CFR|Docket|"
                         r"FR Doc|para\.?|paragraph|section|§)\b", re.I)

CFR    = re.compile(r"\b\d{1,2}\s*CFR\s*(?:[Pp]art\s*)?\d+(?:\.\d+)?")
DOCKET = re.compile(r"\b((?:FSIS|FDA|USDA|APHIS|AMS)-\d{4}-[A-Z0-9]+-?\d*)", re.I)
FRDOC  = re.compile(r"\bFR\s*Doc\.?\s*(?:No\.?)?\s*(\d{4}-\d+)", re.I)
OMB    = re.compile(r"\b(0\d{3}-\d{4})\b")
DATEF  = re.compile(r"\b((?:January|February|March|April|May|June|July|August|September|"
                    r"October|November|December)\s+\d{1,2},?\s+\d{4})", re.I)
NUM    = re.compile(r"(?<![\w./-])(\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)?)(?![\w/-])")
IGNORE_NUMS = {str(i) for i in range(0, 25)} | {"100", "2026", "2025", "1000", "2000"}
ABBREV = re.compile(r"\b(?:No|Nos|Inc|Ltd|Corp|Co|Dr|Mr|Mrs|Ms|St|approx|para|e\.g|i\.e|etc|vs|U\.S|Sec|Fig|Vol)\.", re.I)

SRC_NORM = re.sub(r"\s+", " ", SOURCE_TEXT.replace("\u2013","-").replace("\u2014","-")).lower()

def strip_quotes(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    t = re.sub(r"(?m)^\s*>.*$", " ", t)
    t = re.sub(r"\u201c[^\u201d]{40,}\u201d", " ", t)
    return re.sub(r'"[^"]{40,}"', " ", t)

def sentences(t):
    t = re.sub(r"(?m)^\s*#{1,6}\s.*$", " ", t)
    t = re.sub(r"(?m)^\s*[-*\u2022]\s*", " ", t)
    t = ABBREV.sub(lambda m: m.group(0).replace(".", "\x00"), t)
    out = []
    for p in re.split(r"(?<=[.!?])\s+(?=[A-Z\"'\u201c(])|\n{2,}", t):
        p = p.replace("\x00", ".").strip()
        if p:
            out.append(p)
    return out

def wc(s):
    s = re.sub(r"\([^)]*\)", " ", s)
    return len([w for w in re.sub(r"[^\w\s'-]", " ", s).split() if w.strip("-'")])

def hits(t, pats):
    return [(lab, m.group(0).strip()[:80]) for pat, lab in pats for m in pat.finditer(t)]

def in_source(raw):
    bare = raw.replace(",", "")
    for v in {raw, bare}:
        if re.search(r"(?<![\d,.])" + re.escape(v) + r"(?![\d,.])", SRC_NORM):
            return True
    return bool(re.search(r"(?<![\d.])" + re.escape(bare) + r"(?![\d.])",
                          re.sub(r"(?<=\d),(?=\d)", "", SRC_NORM)))

def fab_candidates(t):
    body, flags = strip_quotes(t), []
    def chk(s, kind):
        n = re.sub(r"\s+", " ", s).lower()
        if n not in SRC_NORM and n.replace(" ", "") not in SRC_NORM.replace(" ", ""):
            flags.append((kind, s.strip()))
    for pat, kind in [(CFR,"CFR cite"),(DOCKET,"docket"),(FRDOC,"FR Doc"),(OMB,"OMB no."),(DATEF,"date")]:
        for m in pat.finditer(body):
            chk(m.group(1) if pat.groups else m.group(0), kind)
    seen = set()
    for m in NUM.finditer(body):
        raw = m.group(1)
        if raw in IGNORE_NUMS or raw in seen:
            continue
        seen.add(raw)
        if not in_source(raw):
            flags.append(("number", raw))
    return flags

MACHINE = ["ai_vocab","contrast","long_sentences","importance_lines","summary_ending",
           "vague_authority","fake_examples","cringe_transitions","unsourced_stats"]
HUMAN   = ["fabrications","citation_failures","voice_drift","structure_rebuild","length_compliance"]
COLS    = ["output"] + MACHINE + HUMAN + ["notes"]

results = []
for p in sorted(OUTPUTS.glob("*.md")):
    text = p.read_text(encoding="utf-8")
    body = strip_quotes(text)
    v = [(w, w) for w in AI_VOCAB for _ in re.finditer(r"\b"+re.escape(w)+r"\b", body, re.I)]
    v += hits(body, CONTEXTUAL)
    longs = [(f"{wc(s)}w", s[:110]) for s in sentences(body) if wc(s) > 16]
    stats = [("unsourced stat", m.group(0)) for s in sentences(body)
             if not (DATE_NEAR.search(s) or SOURCE_NEAR.search(s)) for m in STAT.finditer(s)]
    h = {"ai_vocab": v, "contrast": hits(body, CONTRAST), "long_sentences": longs,
         "importance_lines": hits(body, IMPORTANCE), "summary_ending": hits(text, SUMMARY),
         "vague_authority": hits(body, VAGUE),
         "fake_examples": [("fake example", m.group(0)[:60]) for m in FAKE.finditer(body)],
         "cringe_transitions": hits(body, CRINGE), "unsourced_stats": stats}
    counts = {k: (1 if k == "summary_ending" and val else len(val)) for k, val in h.items()}
    results.append({"output": p.stem, "counts": counts, "hits": h,
                    "fabs": fab_candidates(text), "words": len(text.split())})

with SCORES.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(COLS)
    for r in results:
        w.writerow([r["output"]] + [r["counts"][c] for c in MACHINE] + [""] * len(HUMAN) + [""])

hdr = f"{'output':16s} {'words':>6s} " + " ".join(f"{c[:5]:>5s}" for c in MACHINE) + f" {'MECH':>5s} {'fab?':>5s}"
print(hdr); print("-" * len(hdr))
for r in sorted(results, key=lambda x: x["output"]):
    mech = sum(r["counts"].values())
    print(f"{r['output']:16s} {r['words']:>6d} " +
          " ".join(f"{r['counts'][c]:>5d}" for c in MACHINE) +
          f" {mech:>5d} {len(r['fabs']):>5d}")

lines = ["# Grading report", "", "Machine-counted faults with the triggering text.",
         "Model identity is not in this file; you are still blind.", ""]
for r in sorted(results, key=lambda x: x["output"]):
    lines += [f"## {r['output']}  ({r['words']} words)", ""]
    for col, hh in r["hits"].items():
        if not hh: continue
        lines += [f"**{col}** ({len(hh)})", ""]
        cnt = Counter(l for l, _ in hh)
        for lab, n in cnt.most_common():
            ex = next(e for l, e in hh if l == lab)
            lines.append(f"- `{lab}`{f' x{n}' if n>1 else ''} — {ex}")
        lines.append("")
    if r["fabs"]:
        lines += [f"**fabrication candidates** ({len(r['fabs'])}) — not in the sources; confirm each", ""]
        lines += [f"- `{k}` — {v}" for k, v in r["fabs"]]
        lines.append("")
    lines += ["---", ""]
REPORT.write_text("\n".join(lines), encoding="utf-8")

print(f"\nscores  -> {SCORES}")
print(f"report  -> {REPORT}")
print(f"\n{sum(len(r['fabs']) for r in results)} fabrication candidates. Some are legitimate")
print("(rounded figures, restated math). Confirm each, then fill the human columns.")

## 9. Score by hand

Open `grading_report.md`, read each output, fill five columns in `scores.csv`.

| Column | What | Range |
|---|---|---|
| `fabrications` | confirmed inventions; a rounded figure is not one | count |
| `citation_failures` | citation points at a paragraph that does not say that | count |
| `voice_drift` | 0 sounds like you, 1 slips, 2 press release | 0-2 |
| `structure_rebuild` | 0 ship after trimming, 1 rebuild, 2 start over | 0-2 |
| `length_compliance` | 0 met the 2,000-word floor on A, 1 came in short; 0 on all B rows | 0-1 |

Note on citations: a model that supplies a correct date from training and cites
it to the source document is doing useful analyst work, and your process verifies output
against sources anyway. That is not a fault. An invented docket number is.

Grade one output start to finish before opening the next; side-by-side on the first pass
anchors you to whichever you read first. Brief A on a laptop, Brief B on a phone, since
that is where each gets read.

## 10. Reveal

In [ ]:
key = json.loads(KEY.read_text(encoding="utf-8"))
rows = list(csv.DictReader(SCORES.open(encoding="utf-8")))
merged = [{**r, **key[r["output"].strip()], "output": r["output"].strip()}
          for r in rows if r.get("output", "").strip() in key]

if not merged:
    raise SystemExit("No rows matched key.json — check the `output` column.")

numeric = [c for c in rows[0] if c not in ("output", "notes") and c.strip()]
def faults(r):
    s = 0.0
    for c in numeric:
        try: s += float(r.get(c) or 0)
        except ValueError: pass
    return s
for r in merged:
    r["_f"] = faults(r)

h = f"{'output':16s} {'config':16s} {'eff':>7s} {'rep':>4s} {'faults':>7s} {'words':>6s} {'lat':>7s} {'$/run':>9s}"
print(h); print("-" * len(h))
for r in sorted(merged, key=lambda x: (x["_f"], x["model"])):
    print(f"{r['output']:16s} {r['model']:16s} {str(r.get('effort') or '-'):>7s} "
          f"{int(r.get('rep',1)):>4d} {r['_f']:>7.0f} {r['words']:>6d} "
          f"{r.get('latency_s') or 0:>6.1f}s ${r.get('cost_computed') or 0:>8.4f}")

agg = {}
for r in merged:
    a = agg.setdefault(r["model"], {"f": [], "w": [], "l": [], "c": [], "rt": []})
    a["f"].append(r["_f"]); a["w"].append(r["words"])
    if r.get("latency_s"): a["l"].append(r["latency_s"])
    if r.get("cost_computed"): a["c"].append(r["cost_computed"])
    if r.get("reasoning_tokens"): a["rt"].append(r["reasoning_tokens"])

mean = lambda xs: sum(xs)/len(xs) if xs else 0.0
print(f"\n{'config':16s} {'n':>3s} {'mean':>7s} {'range':>8s} {'words':>7s} {'lat':>8s} {'$/run':>9s} {'rtok':>7s}")
print("-" * 70)
for m, a in sorted(agg.items(), key=lambda kv: mean(kv[1]["f"])):
    rng = f"{min(a['f']):.0f}-{max(a['f']):.0f}" if len(a["f"]) > 1 else "-"
    print(f"{m:16s} {len(a['f']):>3d} {mean(a['f']):>7.1f} {rng:>8s} {mean(a['w']):>7.0f} "
          f"{mean(a['l']):>7.1f}s ${mean(a['c']):>8.4f} {mean(a['rt']):>7.0f}")

spread = [(m, max(a["f"]) - min(a["f"])) for m, a in agg.items() if len(a["f"]) > 1]
if spread:
    worst = max(spread, key=lambda x: x[1])
    if worst[1] > 0:
        gaps = sorted(mean(a["f"]) for a in agg.values())
        between = gaps[-1] - gaps[0]
        print(f"\nWidest spread inside one config: {worst[0]}, {worst[1]:.0f} faults.")
        print(f"Spread between best and worst config: {between:.1f}.")
        if worst[1] >= between:
            print("Within-config noise meets or exceeds between-config difference.")
            print("This benchmark cannot separate these models. The ranking above is noise.")

print("\nWhere two configs sit within a couple of faults, cost decides.")
print("Check current listed rates first; introductory prices expire.")